In [10]:
import rasterio as rio 
import numpy as np 
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

def read_raster(path):
    with rio.open(path) as src:
        return src.read(), src 

# Read the raster file
im_all_bands, src = read_raster('/Data_large/marine/Datasets/VENuS/ds_L0/perfect/ASH_L0_04977_20180711_MultiLayer_mask_OK.tif')

# Check if the source CRS is None
if src.crs is None:
    print("Source CRS is None. Assigning a CRS.")
    # Assign the correct CRS
    src_crs = CRS.from_epsg(4326)  # Replace with the correct EPSG code
else:
    src_crs = src.crs

print("Source CRS:", src_crs)

# Extract the 10th band (bands are zero-indexed in rasterio)
band_index = 9  # 10th band
im = im_all_bands[band_index, :, :]

# Add an axis to match expected shape (1, height, width)
im = im[np.newaxis, :, :]
print("Extracted band shape:", im.shape)

# Reshape into (1, 2048, 2048)
dst_width, dst_height = 2048, 2048

# Calculate the default transform
transform, width, height = calculate_default_transform(
    src_crs, src_crs, src.width, src.height,
    left=src.bounds.left, bottom=src.bounds.bottom, right=src.bounds.right, top=src.bounds.top,
    dst_width=dst_width, dst_height=dst_height
)

# Prepare the destination array
resampled_im = np.empty((1, dst_height, dst_width), dtype=im.dtype)

# Perform the reprojection and resampling
reproject(
    source=im,
    destination=resampled_im,
    src_transform=src.transform,
    src_crs=src_crs,
    dst_transform=transform,
    dst_crs=src_crs,
    resampling=Resampling.bilinear
)

print("Resampled image shape:", resampled_im.shape)

# Write the resampled image to a new file
output_path = '/Data_large/marine/PythonProjects/MMDET/resampled_10th_band.tif'

# Update metadata
new_meta = src.meta.copy()
new_meta.update({
    'driver': 'GTiff',
    'height': dst_height,
    'width': dst_width,
    'transform': transform,
    'crs': src_crs,  # Ensure CRS is set
    'count': 1  # Since we're writing only one band
})

# Write the resampled image
with rio.open(output_path, 'w', **new_meta) as dst:
    dst.write(resampled_im[0], 1)  # Write the first (and only) band

print(f"Resampled image written to {output_path}")

Source CRS is None. Assigning a CRS.
Source CRS: EPSG:4326
Extracted band shape: (1, 2394, 1902)
Resampled image shape: (1, 2048, 2048)
Resampled image written to /Data_large/marine/PythonProjects/MMDET/resampled_10th_band.tif
